In [1]:
import random

class Node:
    def __init__(self, room, parent, action, step, cost):
        self.room = room
        self.parent = parent
        self.action = action
        self.step = step
        self.cost = cost


m = int(input("Nhập số hàng: "))
n = int(input("Nhập số cột: "))

room1 = []
room2 = []

print("Nhập ma trận 1:")
for i in range(m):
    row = list(map(int, input().split()))
    room1.append(row)

print("Nhập ma trận 2:")
for i in range(m):
    row = list(map(int, input().split()))
    room2.append(row)

start_x = random.randint(0, m - 1)
start_y = random.randint(0, n - 1)
start_x2 = random.randint(0, m - 1)
start_y2 = random.randint(0, n - 1)


def room_to_tuple(room):
    return tuple(tuple(row) for row in room)


def is_goal(room):
    for row in room:
        if 1 in row:
            return False
    return True


def print_room(room, x, y):
    for i in range(m):
        for j in range(n):
            if i == x and j == y:
                print("x", end=" ")
            else:
                print(room[i][j], end=" ")

        print()

    print()


def get_rules(x, y, x2, y2):
    moves = []
    if x > 0 or x2 > 0:
        moves.append("UP")

    if x < m - 1 or x2 < m - 1:
        moves.append("DOWN")

    if y > 0 or y2 > 0:
        moves.append("LEFT")

    if y < n - 1 or y2 < n - 1:
        moves.append("RIGHT")

    return moves


def move(room, x, y, action):
    new_room = [row[:] for row in room]
    new_x, new_y = x, y

    if action == "UP" and new_x > 0:
        new_x -= 1

    elif action == "DOWN" and new_x < len(room) - 1:
        new_x += 1

    elif action == "LEFT" and new_y > 0:
        new_y -= 1

    elif action == "RIGHT" and new_y < len(room[0]) - 1:
        new_y += 1

    if new_room[new_x][new_y] == 1:
        new_room[new_x][new_y] = 0

    return new_room, new_x, new_y


def cnt_cost(room):
    cnt = 0
    for i in range(m):
        for j in range(n):
            if room[i][j] == 1:
                cnt += 1
    return cnt


def print_solution(result, start_x, start_x2, start_y, start_y2):
    node, node2 = result
    path = []
    while node is not None and node2 is not None:
        path.append((node, node2))
        node = node.parent
        node2 = node2.parent

    path.reverse()
    x, y = start_x, start_y
    x2, y2 = start_x2, start_y2
    print("Các bước làm sạch phòng:")
    for node, node2 in path:
        if node.action == "STAY":
            pass
        elif node.action:
            _, x, y = move(node.parent.room, x, y, node.action)

        if node2.action == "STAY":
            pass
        elif node2.action:
            _, x2, y2 = move(node2.parent.room, x2, y2, node2.action)
        
        if node.action is None:
            print("Bước 0: Vị trí bắt đầu")
        else:
            print(f"Bước {node.step}: {node.action} | Cost = {node.cost}")

        print_room(node.room, x, y)
        print_room(node2.room, x2, y2)

    print("Tổng số bước:", len(path) - 1)
    print("Tổng cost:", path[-1][0].cost)


def BFS(start_room, start_room2, start_x, start_x2, start_y, start_y2):
    start_room = [row[:] for row in start_room]
    start_room2 = [row[:] for row in start_room2]

    if start_room[start_x][start_y] == 1:
        start_room[start_x][start_y] = 0

    if start_room2[start_x2][start_y2] == 1:
        start_room2[start_x2][start_y2] = 0

    start_node = Node(start_room, None, None, 0, cnt_cost(start_room))
    start_node2 = Node(start_room2, None, None, 0, cnt_cost(start_room2))

    frontier = [(start_node, start_node2, start_x, start_x2, start_y, start_y2)]
    reached = set()
    initial_state = (room_to_tuple(start_room), room_to_tuple(start_room2), start_x, start_x2, start_y, start_y2)
    reached.add(initial_state)

    while frontier:
        current_node, current_node2, x, x2, y, y2 = frontier.pop(0)

        if is_goal(current_node.room) and is_goal(current_node2.room):
            return current_node, current_node2

        for act in get_rules(x, y, x2, y2):
            if is_goal(current_node.room):
                new_room = current_node.room
                new_x = x
                new_y = y
                new_node = Node(new_room, current_node, "STAY", current_node.step + 1, 0)
            else:
                new_room, new_x, new_y = move(current_node.room, x, y, act)
                new_node = Node(new_room, current_node, act, current_node.step + 1, cnt_cost(new_room))

            if is_goal(current_node2.room):
                new_room2 = current_node2.room
                new_x2 = x2
                new_y2 = y2
                new_node2 = Node(new_room2, current_node2, "STAY", current_node2.step + 1, 0)
            else:
                new_room2, new_x2, new_y2 = move(current_node2.room, x2, y2, act)
                new_node2 = Node(new_room2, current_node2, act, current_node2.step + 1, cnt_cost(new_room2))

            new_state = (room_to_tuple(new_room), room_to_tuple(new_room2), new_x, new_x2, new_y, new_y2)

            if new_state not in reached:
                reached.add(new_state)
                frontier.append((new_node, new_node2, new_x, new_x2, new_y, new_y2))

    return None


print("Vị trí bắt đầu của máy hút bụi 1:", (start_x, start_y))
print("Vị trí bắt đầu của máy hút bụi 2:", (start_x2, start_y2))
print()

result = BFS(room1, room2, start_x, start_x2, start_y, start_y2)

if result:
    print_solution(result, start_x, start_x2, start_y, start_y2)
else:
    print("Không tìm được lời giải")


Nhập ma trận 1:
Nhập ma trận 2:
Vị trí bắt đầu của máy hút bụi 1: (0, 0)
Vị trí bắt đầu của máy hút bụi 2: (1, 0)

Các bước làm sạch phòng:
Bước 0: Vị trí bắt đầu
x 1 1 
1 0 1 
1 1 0 

1 0 1 
x 1 0 
1 0 1 

Bước 1: UP | Cost = 6
x 1 1 
1 0 1 
1 1 0 

x 0 1 
0 1 0 
1 0 1 

Bước 2: DOWN | Cost = 5
0 1 1 
x 0 1 
1 1 0 

0 0 1 
x 1 0 
1 0 1 

Bước 3: DOWN | Cost = 4
0 1 1 
0 0 1 
x 1 0 

0 0 1 
0 1 0 
x 0 1 

Bước 4: RIGHT | Cost = 3
0 1 1 
0 0 1 
0 x 0 

0 0 1 
0 1 0 
0 x 1 

Bước 5: UP | Cost = 3
0 1 1 
0 x 1 
0 0 0 

0 0 1 
0 x 0 
0 0 1 

Bước 6: UP | Cost = 2
0 x 1 
0 0 1 
0 0 0 

0 x 1 
0 0 0 
0 0 1 

Bước 7: RIGHT | Cost = 1
0 0 x 
0 0 1 
0 0 0 

0 0 x 
0 0 0 
0 0 1 

Bước 8: DOWN | Cost = 0
0 0 0 
0 0 x 
0 0 0 

0 0 0 
0 0 x 
0 0 1 

Bước 9: STAY | Cost = 0
0 0 0 
0 0 x 
0 0 0 

0 0 0 
0 0 0 
0 0 x 

Tổng số bước: 9
Tổng cost: 0
